# 导入依赖和文件

In [33]:
import json
import joblib
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

In [34]:

# 加载特征数据
X = np.load("X.npy")
X_test = np.load("X_test.npy")
y = np.load("y.npy")

# 划分训练集与验证集

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, train_size=0.8, test_size=0.2, random_state=0)


# 寻找并保存最优参数

In [35]:

# 定义 LGBM 的目标函数
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 6),
        'num_leaves': trial.suggest_int('num_leaves', 20, 30),
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),
        'random_state': 0
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 定义 CatBoost 的目标函数
def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 6),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 3),
        'border_count': trial.suggest_categorical('border_count', [32, 64]),
        'verbose': False,
        'random_state': 0
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 创建 Optuna 研究对象并进行优化
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=50)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=50)

# 获取最优参数
lgbm_best_params = lgbm_study.best_params
lgbm_best_value = lgbm_study.best_value
catboost_best_params = catboost_study.best_params
catboost_best_value = catboost_study.best_value

[I 2025-04-27 16:50:05,597] A new study created in memory with name: no-name-5211743d-8b45-42a5-8f51-2b42ed868eeb
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:50:05,672] Trial 0 finished with value: 0.7866589994249569 and parameters: {'n_estimators': 100, 'learning_rate': 0.02020042400432104, 'max_depth': 6, 'num_leaves': 27, 'subsample': 0.8731250141719037, 'colsample_bytree': 0.9003173584153767}. Best is trial 0 with value: 0.7866589994249569.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:50:05,707] Trial 1 finished with value: 0.7745830937320299 and parameters: {'n_estimators': 50, '

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000536 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000312 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:50:05,854] Trial 4 finished with value: 0.7981598619896493 and parameters: {'n_estimators': 100, 'learning_rate': 0.07181142912684059, 'max_depth': 5, 'num_leaves': 29, 'subsample': 0.941228521973758, 'colsample_bytree': 0.8248146644879708}. Best is trial 4 with value: 0.7981598619896493.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:50:05,887] Trial 5 finished with value: 0.7722829212190915 and parameters: {'n_estimators': 50, 'learning_rate': 0.011654489317267806, 'max_depth': 5, 'num_leaves': 21, 'subsample': 0.9314679997922103, 'colsample

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000314 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:50:06,107] Trial 8 finished with value: 0.7837837837837838 and parameters: {'n_estimators': 100, 'learning_rate': 0.014224846673956563, 'max_depth': 5, 'num_leaves': 24, 'subsample': 0.8668991337111512, 'colsample_bytree': 0.8854447164808272}. Best is trial 4 with value: 0.7981598619896493.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:50:06,144] Trial 9 finished with value: 0.7809085681426107 and parameters: {'n_estimators': 50, 'learning_rate': 0.03457756017879091, 'max_depth': 4, 'num_leaves': 29, 'subsample': 0.8502747091788666, 'colsampl

...

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:50:08,421] Trial 45 finished with value: 0.8004600345025877 and parameters: {'n_estimators': 100, 'learning_rate': 0.07749703602145089, 'max_depth': 6, 'num_leaves': 22, 'subsample': 0.8357548959692108, 'colsample_bytree': 0.826420696725069}. Best is trial 31 with value: 0.8108108108108109.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:50:08,467] Trial 46 finished with value: 0.7958596894767107 and parameters: {'n_estimators': 50, 'learning_rate': 0.08838593933868878, 'max_depth': 6, 'num_leaves': 24, 'subsample': 0.824800744773633, 'colsampl

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:50:08,659] Trial 49 finished with value: 0.8021851638872916 and parameters: {'n_estimators': 100, 'learning_rate': 0.06576621855382546, 'max_depth': 6, 'num_leaves': 21, 'subsample': 0.8100979957562556, 'colsample_bytree': 0.8092843235493256}. Best is trial 31 with value: 0.8108108108108109.
[I 2025-04-27 16:50:08,660] A new study created in memory with name: no-name-f16a5c9d-f9ca-48b4-b3f9-ad6b5c42b85b


[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000450 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

[I 2025-04-27 16:50:08,891] Trial 0 finished with value: 0.7883841288096607 and parameters: {'n_estimators': 100, 'learning_rate': 0.02310829222398401, 'depth': 6, 'l2_leaf_reg': 3, 'border_count': 64}. Best is trial 0 with value: 0.7883841288096607.
[I 2025-04-27 16:50:09,142] Trial 1 finished with value: 0.79700977573318 and parameters: {'n_estimators': 100, 'learning_rate': 0.07080652752432577, 'depth': 6, 'l2_leaf_reg': 2, 'border_count': 64}. Best is trial 1 with value: 0.79700977573318.
[I 2025-04-27 16:50:09,285] Trial 2 finished with value: 0.7889591719378953 and parameters: {'n_estimators': 50, 'learning_rate': 0.09315742669748511, 'depth': 4, 'l2_leaf_reg': 3, 'border_count': 64}. Best is trial 1 with value: 0.79700977573318.
[I 2025-04-27 16:50:09,464] Trial 3 finished with value: 0.7814836112708453 and parameters: {'n_estimators': 50, 'learning_rate': 0.03010440638766118, 'depth': 6, 'l2_leaf_reg': 3, 'border_count': 64}. Best is trial 1 with value: 0.79700977573318.
[I 202

In [36]:
print("LGBM 最佳得分:", lgbm_best_value)
print("CatBoost 最佳得分:", catboost_best_value)
# 导出 LGBM 和 CatBoost 的最优参数到 JSON 文件
with open('lgbm_best_params.json', 'w') as f:
    json.dump(lgbm_best_params, f)

with open('catboost_best_params.json', 'w') as f:
    json.dump(catboost_best_params, f)

LGBM 最佳得分: 0.8108108108108109
CatBoost 最佳得分: 0.7981598619896493


# 训练并保存模型

In [37]:

# 导入最优参数
with open('lgbm_best_params.json', 'r') as f:
    lgbm_best_params = json.load(f)

with open('catboost_best_params.json', 'r') as f:
    catboost_best_params = json.load(f)

# 使用最优参数重新实例化模型并训练
lgbm_model = LGBMClassifier(**lgbm_best_params, random_state=0)
catboost_model = CatBoostClassifier(**catboost_best_params, verbose=False, random_state=0)

lgbm_model.fit(X_train, y_train)
catboost_model.fit(X_train, y_train)

joblib.dump(lgbm_model, 'lgbm_best_model.joblib')
joblib.dump(catboost_model, 'catboost_best_model.joblib')


[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000549 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

['catboost_best_model.joblib']

In [46]:
# 加载模型
lgbm_loaded_model = joblib.load('lgbm_best_model.joblib')
catboost_loaded_model = joblib.load('catboost_best_model.joblib')

# 对测试数据进行预测，获取概率值
lgbm_prob = lgbm_model.predict_proba(X_test)[:, 1]
catboost_prob = catboost_model.predict_proba(X_test)[:, 1]

# 简单平均融合
ensemble_prob = (lgbm_prob + catboost_prob) / 2

# 根据阈值生成最终预测
threshold = 0.5
ensemble_pred = ensemble_prob > threshold

# 加载测试数据
test_data = pd.read_csv('test.csv')  # 确保文件路径正确

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],  # 确保这里使用正确的列名
    'Transported': ensemble_pred
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)


/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# 模型融合与提交文件

基学习器已经训练好，现在使用元学习器进行模型融合

In [38]:
# # 加载训练好的基模型
# lgbm_model = joblib.load('lgbm_best_model.joblib')
# catboost_model = joblib.load('catboost_best_model.joblib')

# # 检查模型加载是否正确
# print(lgbm_model.get_params())
# print(catboost_model.get_params())

# # 定义生成元特征的函数
# def generate_meta_features(model, X_train, y_train, X_valid, y_valid, X_test, n_splits=5):
#     print("Entering generate_meta_features function")  # 调试信息
#     kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
#     meta_train = np.zeros((X_train.shape[0],))
#     meta_valid = np.zeros((X_valid.shape[0],))
#     meta_test = np.zeros((X_test.shape[0],))
    
#     for train_index, val_index in kf.split(X_train):
#         print("Inside KFold loop")  # 调试信息
#         X_tr, X_val = X_train[train_index], X_train[val_index]
#         y_tr = y_train[train_index]  # 获取当前折的训练目标变量
        
#         # 确保 X_tr 和 y_tr 的维度正确
#         print(f"X_tr shape: {X_tr.shape}, y_tr shape: {y_tr.shape}")
        
#         model.fit(X_tr, y_tr)  # 传递 y_tr 作为目标变量
#         meta_train[val_index] = model.predict_proba(X_val)[:, 1]
#         print("Completed a fold")  # 调试信息
    
#     # 使用整个训练集重新训练模型以生成验证集和测试集的元特征
#     print("Refitting model on entire training set")  # 调试信息
#     model.fit(X_train, y_train)
#     meta_valid = model.predict_proba(X_valid)[:, 1]
#     meta_test = model.predict_proba(X_test)[:, 1]
#     print("Exiting generate_meta_features function")  # 调试信息
    
#     return meta_train, meta_valid, meta_test

# # 为 LGBM 和 CatBoost 生成元特征
# print("Generating meta features for LGBM")  # 调试信息
# lgbm_meta_train, lgbm_meta_valid, lgbm_meta_test = generate_meta_features(lgbm_model, X_train, y_train, X_valid, y_valid, X_test)
# print("Generating meta features for CatBoost")  # 调试信息
# catboost_meta_train, catboost_meta_valid, catboost_meta_test = generate_meta_features(catboost_model, X_train, y_train, X_valid, y_valid, X_test)

# # 构建元特征矩阵
# X_train_meta = np.column_stack((lgbm_meta_train, catboost_meta_train))
# X_valid_meta = np.column_stack((lgbm_meta_valid, catboost_meta_valid))
# X_test_meta = np.column_stack((lgbm_meta_test, catboost_meta_test))

# # 定义元模型的目标函数
# def meta_objective(trial):
#     params = {
#         'C': trial.suggest_float('C', 0.01, 10.0, log=True),
#         'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']),
#         'random_state': 0
#     }
#     model = LogisticRegression(**params)
#     model.fit(X_train_meta, y_train)
#     y_pred = model.predict(X_valid_meta)
#     accuracy = accuracy_score(y_valid, y_pred)
#     print(f"Meta model accuracy: {accuracy:.4f}")  # 调试信息
#     return accuracy

# # 创建 Optuna 研究对象并进行优化
# print("Starting Optuna study for meta model")  # 调试信息
# meta_study = optuna.create_study(direction='maximize')
# meta_study.optimize(meta_objective, n_trials=50)

# # 获取最优参数
# meta_best_params = meta_study.best_params
# meta_best_value = meta_study.best_value

# print(f"元模型最优参数: {meta_best_params}")
# print(f"元模型最优准确率: {meta_best_value:.4f}")

# # 使用最优参数训练最终的元模型
# best_meta_model = LogisticRegression(**meta_best_params, random_state=0)
# best_meta_model.fit(X_train_meta, y_train)

# # 保存元模型
# joblib.dump(best_meta_model, 'best_meta_model.joblib')


In [43]:
# 加载训练好的基模型
lgbm_model = joblib.load('lgbm_best_model.joblib')
catboost_model = joblib.load('catboost_best_model.joblib')

# 检查模型加载是否正确
print("LGBM Model Parameters:", lgbm_model.get_params())
print("CatBoost Model Parameters:", catboost_model.get_params())

# 定义生成元特征的函数
def generate_meta_features(model, X_train, y_train, X_valid, y_valid, X_test, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    meta_train = np.zeros((X_train.shape[0],))
    meta_valid = np.zeros((X_valid.shape[0],))
    meta_test = np.zeros((X_test.shape[0],))
    
    for train_index, val_index in kf.split(X_train):
        X_tr, X_val = X_train[train_index], X_train[val_index]
        y_tr = y_train[train_index]
        model.fit(X_tr, y_tr)
        meta_train[val_index] = model.predict_proba(X_val)[:, 1]
    
    # 使用整个训练集重新训练模型以生成验证集和测试集的元特征
    model.fit(X_train, y_train)
    meta_valid = model.predict_proba(X_valid)[:, 1]
    meta_test = model.predict_proba(X_test)[:, 1]
    
    return meta_train, meta_valid, meta_test


print("Generating meta features for LGBM")
lgbm_meta_train, lgbm_meta_valid, lgbm_meta_test = generate_meta_features(lgbm_model, X_train, y_train, X_valid, y_valid, X_test)

print("Generating meta features for CatBoost")
catboost_meta_train, catboost_meta_valid, catboost_meta_test = generate_meta_features(catboost_model, X_train, y_train, X_valid, y_valid, X_test)

# 构建元特征矩阵
X_train_meta = np.column_stack((lgbm_meta_train, catboost_meta_train))
X_valid_meta = np.column_stack((lgbm_meta_valid, catboost_meta_valid))
X_test_meta = np.column_stack((lgbm_meta_test, catboost_meta_test))

# 定义元学习器的超参数优化目标函数
def logistic_regression_objective(trial):
    params = {
        'C': trial.suggest_float('C', 0.01, 10.0, log=True),  # 调整范围：更广泛的正则化强度
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']),
        'random_state': 0
    }
    model = LogisticRegression(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def random_forest_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),  # 增加上限
        'max_depth': trial.suggest_int('max_depth', 5, 20),  # 增加上限
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),  # 增加上限
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),  # 增加上限
        'random_state': 0
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),  # 增加上限
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),  # 保持不变
        'max_depth': trial.suggest_int('max_depth', 5, 20),  # 增加上限
        'num_leaves': trial.suggest_int('num_leaves', 10, 200),  # 增加上限
        'random_state': 0
    }
    model = LGBMClassifier(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),  # 增加上限
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),  # 保持不变
        'max_depth': trial.suggest_int('max_depth', 5, 16),  # 增加上限
        'random_state': 0
    }
    model = CatBoostClassifier(**params, verbose=0)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def lasso_objective(trial):
    params = {
        'alpha': trial.suggest_float('alpha', 0.0001, 10.0, log=True),  # 增加上限
        'random_state': 0
    }
    model = Lasso(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    y_pred_class = (y_pred > 0.5).astype(int)
    return accuracy_score(y_valid, y_pred_class)

# 创建Optuna研究对象并进行优化
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=100)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=100)

logistic_regression_study = optuna.create_study(direction='maximize')
logistic_regression_study.optimize(logistic_regression_objective, n_trials=100)

random_forest_study = optuna.create_study(direction='maximize')
random_forest_study.optimize(random_forest_objective, n_trials=100)

lasso_study = optuna.create_study(direction='maximize')
lasso_study.optimize(lasso_objective, n_trials=100)

# 获取最优参数和准确率
meta_models = {
    'LogisticRegression': {
        'best_params': logistic_regression_study.best_params,
        'best_accuracy': logistic_regression_study.best_value
    },
    'RandomForest': {
        'best_params': random_forest_study.best_params,
        'best_accuracy': random_forest_study.best_value
    },
    'LGBMClassifier': {
        'best_params': lgbm_study.best_params,
        'best_accuracy': lgbm_study.best_value
    },
    'CatBoostClassifier': {
        'best_params': catboost_study.best_params,
        'best_accuracy': catboost_study.best_value
    },
    'LassoRegression': {
        'best_params': lasso_study.best_params,
        'best_accuracy': lasso_study.best_value
    }
}


# 选择最佳元学习器
best_meta_model_name = max(meta_models, key=lambda k: meta_models[k]['best_accuracy'])
best_meta_model_params = meta_models[best_meta_model_name]['best_params']
best_meta_model_accuracy = meta_models[best_meta_model_name]['best_accuracy']

# 根据最佳元学习器的名称创建模型实例并训练
if best_meta_model_name == 'LogisticRegression':
    best_meta_model = LogisticRegression(**best_meta_model_params, random_state=0)
elif best_meta_model_name == 'RandomForest':
    best_meta_model = RandomForestClassifier(**best_meta_model_params, random_state=0)
elif best_meta_model_name == 'LGBMClassifier':
    best_meta_model = LGBMClassifier(**best_meta_model_params, random_state=0)
elif best_meta_model_name == 'CatBoostClassifier':
    best_meta_model = CatBoostClassifier(**best_meta_model_params, random_state=0, verbose=0)
elif best_meta_model_name == 'LassoRegression':
    best_meta_model = Lasso(**best_meta_model_params, random_state=0)

# 如果是Lasso回归，需要将预测结果转换为分类标签
if best_meta_model_name == 'LassoRegression':
    best_meta_model.fit(X_train_meta, y_train)
    y_pred_final = best_meta_model.predict(X_valid_meta)
    y_pred_final_class = (y_pred_final > 0.5).astype(int)
    final_accuracy = accuracy_score(y_valid, y_pred_final_class)
    final_report = classification_report(y_valid, y_pred_final_class)
else:
    best_meta_model.fit(X_train_meta, y_train)
    y_pred_final = best_meta_model.predict(X_valid_meta)
    final_accuracy = accuracy_score(y_valid, y_pred_final)
    final_report = classification_report(y_valid, y_pred_final)

# 保存元模型
joblib.dump(best_meta_model, 'best_meta_model.joblib')



LGBM Model Parameters: {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.8742581414585876, 'importance_type': 'split', 'learning_rate': 0.09953787244395743, 'max_depth': 6, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 22, 'objective': None, 'random_state': 0, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.8604796427689769, 'subsample_for_bin': 200000, 'subsample_freq': 0}
CatBoost Model Parameters: {'learning_rate': 0.09485193399699214, 'depth': 5, 'l2_leaf_reg': 1, 'border_count': 32, 'verbose': False, 'n_estimators': 100, 'random_state': 0}
Generating meta features for LGBM
[LightGBM] [Info] Number of positive: 2802, number of negative: 2761
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000413 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1950
[LightGBM] [Info] Number of data points in the

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site

[LightGBM] [Info] Number of positive: 2815, number of negative: 2749
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000253 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1948
[LightGBM] [Info] Number of data points in the train set: 5564, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.505931 -> initscore=0.023725
[LightGBM] [Info] Start training from score 0.023725
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:59:06,474] A new study created in memory with name: no-name-206517f7-6ec0-433a-815c-4aef23e6a93e
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:59:06,523] Trial 0 finished with value: 0.7837837837837838 and parameters: {'n_estimators': 202, 'learning_rate': 0.23277095906725478, 'max_depth': 6, 'num_leaves': 140}. Be

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000053 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

...

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000057 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000052 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No fu

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:59:28,473] Trial 98 finished with value: 0.7596319723979299 and parameters: {'n_estimators': 585, 'learning_rate': 0.21071140316972056, 'max_depth': 10, 'num_leaves': 32}. Best is trial 80 with value: 0.8016101207590569.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 16:59:28,671] Trial 99 finished with value: 0.7975848188614146 and parameters: {'n_estimators': 545, 'learning_rate': 0.048068406854156615, 'max_depth': 12, 'num_leaves': 10}. Best is trial 80 with value: 0.8016101207590569.
[I 2025-04-27 16:59:28,671] A new study created in memory with name: no-name-cf10a443-54f8-4321-8261-c8654d42e014
[I 2025-04-27 16:59:47,347] Trial 0 finished with value: 0.7780333525014376 and parameters: {'n_estimators': 404, 'learning_rate': 0.13085033187013337, 'max_depth': 14}. Best is trial 0 with value: 0.7780333525014376.
[I 2025-04-27 16:59:54,277] Trial 1 finished with value: 0.7993099482461185 and parameters: {'n_estimators': 300, 'learning_rate': 0.024680957647558517, 'max_depth': 13}. Best is trial 1 with value

['best_meta_model.joblib']

In [44]:
# 保存最佳参数到JSON文件
best_params = {
    'LogisticRegression': logistic_regression_study.best_params,
    'RandomForest': random_forest_study.best_params,
    'LGBMClassifier': lgbm_study.best_params,
    'CatBoostClassifier': catboost_study.best_params,
    'LassoRegression': lasso_study.best_params
}

with open('best_meta_params.json', 'w') as f:
    json.dump(best_params, f, indent=4)

In [40]:
for name, metrics in meta_models.items():
    print(f"{name}: Best Accuracy = {metrics['best_accuracy']:.4f}, Best Params = {metrics['best_params']}")
print(f"\n{best_meta_model} Evaluation:")
print(f"Accuracy: {final_accuracy:.4f}")
print(f"Classification Report:\n{final_report}")

LogisticRegression: Best Accuracy = 0.8062, Best Params = {'C': 0.05551064966031306, 'solver': 'newton-cg'}
RandomForest: Best Accuracy = 0.8051, Best Params = {'n_estimators': 488, 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 1}
LGBMClassifier: Best Accuracy = 0.8045, Best Params = {'n_estimators': 192, 'learning_rate': 0.09922343292732684, 'max_depth': 3, 'num_leaves': 22}
CatBoostClassifier: Best Accuracy = 0.8056, Best Params = {'n_estimators': 172, 'learning_rate': 0.018953235644972193, 'max_depth': 10}
LassoRegression: Best Accuracy = 0.8120, Best Params = {'alpha': 0.020470733003659595}

Lasso(alpha=0.020470733003659595, random_state=0) Evaluation:
Accuracy: 0.8120
Classification Report:
              precision    recall  f1-score   support

       False       0.83      0.79      0.81       863
        True       0.80      0.84      0.82       876

    accuracy                           0.81      1739
   macro avg       0.81      0.81      0.81      1739
weighted

In [45]:
# 加载元模型
best_meta_model = joblib.load('best_meta_model.joblib')

# 生成预测结果
pred = best_meta_model.predict(X_test_meta)

# 加载测试数据
test_data = pd.read_csv('test.csv')

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Transported': pred > 0.5
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)